# Corpus Visualizer — mod/onfalo
**Tenant:** `scriptorium` · **Database:** `mod-onfalo`

Proyecta los embeddings de las colecciones Chroma de `mod/onfalo` en 2D/3D usando UMAP.

Estado actual de colecciones activas:

| Colección | Proyecto | Piezas |
|-----------|----------|--------|
| `mo_regulacion_discursos` | REGULACION | WGS Dubai |
| `mo_regulacion_articulos` | REGULACION | docs 07, 09, 09a |
| `mo_regulacion_fichas` | REGULACION | docs 09b, 13 |
| `mo_regulacion_propuestas` | REGULACION | docs 12, 10c, España Crece |
| `mo_regulacion_cartas` | REGULACION | docs 10k, 11, 08 |

Descomenta las secciones siguientes cuando se ingesten los otros 4 proyectos.

**Requisitos:** `pip install chromadb umap-learn plotly pandas numpy scikit-learn`

In [ ]:
%pip install -q chromadb umap-learn plotly pandas numpy scikit-learn "nbformat>=4.2.0"

In [ ]:
import chromadb
import numpy as np
import pandas as pd
import umap
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
from sklearn.preprocessing import LabelEncoder


def show(fig):
    """Renderiza una figura Plotly en VS Code sin depender de nbformat."""
    display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))


from pathlib import Path
STORAGE_PATH = str(Path.home() / 'OASIS' / 'aleph-scriptorium' / 'ARCHIVO' / 'PLUGINS' / 'VECTOR_MACHINE' / 'STORAGE')

client = chromadb.PersistentClient(path=STORAGE_PATH)
print('Colecciones disponibles:', [c.name for c in client.list_collections()])

## Carga de embeddings

Editar `COLLECTIONS` para activar/desactivar proyectos según lo que esté ingestado.

In [ ]:
# ── REGULACION (activo) ──────────────────────────────────────────────────────
COLLECTIONS = [
    'mo_regulacion_discursos',
    'mo_regulacion_articulos',
    'mo_regulacion_fichas',
    'mo_regulacion_propuestas',
    'mo_regulacion_cartas',
]

# ── ATLAS (descomentar cuando esté ingestado) ────────────────────────────────
# COLLECTIONS += [
#     'mo_atlas_geografia',
#     'mo_atlas_historia',
#     'mo_atlas_soberania',
#     'mo_atlas_personajes',
#     'mo_atlas_cronica',
# ]

# ── MAPAS_DE_SINGULARIDAD (descomentar cuando esté ingestado) ────────────────
# COLLECTIONS += [
#     'mo_mapas_corpus',
#     'mo_mapas_geopolitica',
#     'mo_mapas_arquitectura',
#     'mo_mapas_abstract',
# ]

# ── ESCANO_SINTETICO (descomentar cuando esté ingestado) ─────────────────────
# COLLECTIONS += [
#     'mo_escano_whitepaper',
#     'mo_escano_transcripcion',
#     'mo_escano_diagnostico',
#     'mo_escano_delta',
# ]

# ── PROYECTO_DECOHERENCIA (descomentar cuando esté ingestado) ────────────────
# COLLECTIONS += [
#     'mo_decoherencia_corpus',
#     'mo_decoherencia_base',
#     'mo_decoherencia_ratio',
#     'mo_decoherencia_sesiones',
# ]

LABELS = {
    # REGULACION
    'mo_regulacion_discursos':  'Discursos [RG]',
    'mo_regulacion_articulos':  'Artículos [RG]',
    'mo_regulacion_fichas':     'Fichas [RG]',
    'mo_regulacion_propuestas': 'Propuestas [RG]',
    'mo_regulacion_cartas':     'Cartas [RG]',
    # ATLAS
    'mo_atlas_geografia':    'Geografía [AT]',
    'mo_atlas_historia':     'Historia [AT]',
    'mo_atlas_soberania':    'Soberanía [AT]',
    'mo_atlas_personajes':   'Personajes [AT]',
    'mo_atlas_cronica':      'Crónica [AT]',
    # MAPAS
    'mo_mapas_corpus':        'Corpus [MS]',
    'mo_mapas_geopolitica':   'Geopolítica [MS]',
    'mo_mapas_arquitectura':  'Arquitectura [MS]',
    'mo_mapas_abstract':      'Abstract [MS]',
    # ESCANO
    'mo_escano_whitepaper':    'Whitepaper [ES]',
    'mo_escano_transcripcion': 'Transcripción [ES]',
    'mo_escano_diagnostico':   'Diagnóstico [ES]',
    'mo_escano_delta':         'Delta [ES]',
    # DECOHERENCIA
    'mo_decoherencia_corpus':   'Corpus [DC]',
    'mo_decoherencia_base':     'Base [DC]',
    'mo_decoherencia_ratio':    'Ratio [DC]',
    'mo_decoherencia_sesiones': 'Sesiones [DC]',
}

rows = []
for col_name in COLLECTIONS:
    col = client.get_collection(col_name)
    result = col.get(include=['embeddings', 'documents', 'metadatas'])
    for doc_id, emb, doc, meta in zip(
        result['ids'], result['embeddings'], result['documents'], result['metadatas']
    ):
        rows.append({
            'id':        doc_id,
            'coleccion': LABELS.get(col_name, col_name),
            'col_raw':   col_name,
            'bloque':    meta.get('bloque', '?'),
            'tipo':      meta.get('tipo', '?'),
            'marca':     meta.get('marca', doc_id),
            'texto':     doc[:120] + '...' if len(doc) > 120 else doc,
            'embedding': emb,
        })

df = pd.DataFrame(rows)
embeddings_matrix = np.array(df['embedding'].tolist())
print(f'Total piezas cargadas: {len(df)}')
print(df.groupby('coleccion').size().to_string())

## Proyección UMAP — 2D y 3D

In [ ]:
UMAP_PARAMS = dict(n_neighbors=5, min_dist=0.2, random_state=42, low_memory=False)

reducer_2d = umap.UMAP(n_components=2, **UMAP_PARAMS)
proj_2d = reducer_2d.fit_transform(embeddings_matrix)
df['x'] = proj_2d[:, 0]
df['y'] = proj_2d[:, 1]

reducer_3d = umap.UMAP(n_components=3, **UMAP_PARAMS)
proj_3d = reducer_3d.fit_transform(embeddings_matrix)
df['x3'] = proj_3d[:, 0]
df['y3'] = proj_3d[:, 1]
df['z3'] = proj_3d[:, 2]

print('Proyección 2D shape:', proj_2d.shape)
print('Proyección 3D shape:', proj_3d.shape)

## Vista 2D — paleta índigo

In [ ]:
# Paleta azul índigo — mod/onfalo
COLOR_MAP = {
    # REGULACION
    'Discursos [RG]':  '#7ec8e3',  # cyan claro
    'Artículos [RG]':  '#3d85c8',  # azul medio
    'Fichas [RG]':     '#6aa3d6',  # azul-gris
    'Propuestas [RG]': '#1a3a8f',  # índigo profundo
    'Cartas [RG]':     '#4a6fa5',  # azul pizarra
    # ATLAS (reservados)
    'Geografía [AT]':  '#a8d8ea',
    'Historia [AT]':   '#89b4d1',
    'Soberanía [AT]':  '#5b9dc9',
    'Personajes [AT]': '#2e6da4',
    'Crónica [AT]':    '#1b4f80',
    # MAPAS (reservados)
    'Corpus [MS]':       '#c3e0f3',
    'Geopolítica [MS]':  '#8cbedd',
    'Arquitectura [MS]': '#5596c4',
    'Abstract [MS]':     '#2b6fa3',
    # ESCANO (reservados)
    'Whitepaper [ES]':    '#d4eaf7',
    'Transcripción [ES]': '#9ecae1',
    'Diagnóstico [ES]':   '#6baed6',
    'Delta [ES]':         '#3182bd',
    # DECOHERENCIA (reservados)
    'Corpus [DC]':   '#deebf7',
    'Base [DC]':     '#c6dbef',
    'Ratio [DC]':    '#9ecae1',
    'Sesiones [DC]': '#6baed6',
}

BG = '#07101f'

fig2d = px.scatter(
    df, x='x', y='y',
    color='coleccion',
    color_discrete_map=COLOR_MAP,
    hover_data={'marca': True, 'tipo': True, 'bloque': True, 'texto': True, 'x': False, 'y': False},
    text='marca',
    title='mod/onfalo — Espacio de embeddings 2D (UMAP)',
    width=950, height=680,
)
fig2d.update_traces(textposition='top center', textfont_size=9, marker=dict(size=10, opacity=0.85))
fig2d.update_layout(
    legend_title_text='Colección',
    plot_bgcolor=BG,
    paper_bgcolor=BG,
    font_color='#c8d8e8',
)
show(fig2d)

## Vista 3D — paleta índigo

In [ ]:
fig3d = px.scatter_3d(
    df, x='x3', y='y3', z='z3',
    color='coleccion',
    color_discrete_map=COLOR_MAP,
    hover_data={'marca': True, 'tipo': True, 'bloque': True, 'texto': True,
                'x3': False, 'y3': False, 'z3': False},
    text='marca',
    title='mod/onfalo — Espacio de embeddings 3D (UMAP)',
    width=950, height=720,
)
fig3d.update_traces(textposition='top center', textfont_size=8, marker=dict(size=5, opacity=0.9))
fig3d.update_layout(
    legend_title_text='Colección',
    paper_bgcolor=BG,
    font_color='#c8d8e8',
    scene=dict(
        bgcolor=BG,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    ),
)
show(fig3d)

## Análisis de clusters automático (KMeans)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings_matrix)
    scores[k] = silhouette_score(embeddings_matrix, labels)

best_k = max(scores, key=scores.get)
print(f'Silhouette scores: {scores}')
print(f'K óptimo: {best_k} (score={scores[best_k]:.3f})')

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = km_best.fit_predict(embeddings_matrix).astype(str)

fig_cl = px.scatter(
    df, x='x', y='y',
    color='cluster',
    hover_data={'marca': True, 'coleccion': True, 'tipo': True, 'texto': True},
    text='marca',
    title=f'mod/onfalo — Clusters KMeans (k={best_k}) sobre 2D UMAP',
    width=950, height=680,
)
fig_cl.update_traces(textposition='top center', textfont_size=9, marker=dict(size=10, opacity=0.8))
fig_cl.update_layout(plot_bgcolor=BG, paper_bgcolor=BG, font_color='#c8d8e8')
show(fig_cl)

## Divergencia corpus ↔ geometría

Detecta piezas que el Archivero clasificó en una colección pero que geométricamente
son más cercanas a piezas de otra — posible tensión semántica no capturada en texto.

In [ ]:
from sklearn.neighbors import NearestNeighbors

nbrs = NearestNeighbors(n_neighbors=4, metric='cosine').fit(embeddings_matrix)
distances, indices = nbrs.kneighbors(embeddings_matrix)

divergencias = []
for i, (dists_i, idxs_i) in enumerate(zip(distances, indices)):
    pieza = df.iloc[i]
    vecinos = df.iloc[idxs_i[1:]]
    vecinos_col = vecinos['col_raw'].value_counts()
    col_mayoritaria = vecinos_col.index[0]
    if col_mayoritaria != pieza['col_raw'] and vecinos_col.iloc[0] >= 2:
        divergencias.append({
            'marca':          pieza['marca'],
            'col_asignada':   pieza['coleccion'],
            'col_geometrica': LABELS.get(col_mayoritaria, col_mayoritaria),
            'vecinos':        list(vecinos['marca']),
            'dist_media':     round(float(dists_i[1:].mean()), 4),
        })

if divergencias:
    print(f'Piezas con divergencia corpus ↔ geometría ({len(divergencias)}):\n')
    for d in divergencias:
        print(f"  {d['marca']}")
        print(f"    Asignada:   {d['col_asignada']}")
        print(f"    Geometría:  {d['col_geometrica']}")
        print(f"    Vecinos:    {d['vecinos']}")
        print(f"    Dist media: {d['dist_media']}\n")
else:
    print('Sin divergencias — la taxonomía del Archivero coincide con la geometría.')


## Ejemplo de query semántica

In [ ]:
# ── Query semántica sobre las colecciones activas de mod/onfalo ───────────────
# Edita QUERY y COLECCIONES_QUERY para explorar el espacio vectorial.

QUERY = '¿Quién regula a los reguladores de la IA?'
COLECCIONES_QUERY = COLLECTIONS   # usa las colecciones activas del bloque de carga
N_RESULTADOS = 3

print(f'Query: «{QUERY}»\n')
print('=' * 70)

for col_name in COLECCIONES_QUERY:
    col = client.get_collection(col_name)
    result = col.query(
        query_texts=[QUERY],
        n_results=N_RESULTADOS,
        include=['documents', 'metadatas', 'distances'],
    )
    docs      = result['documents'][0]
    metas     = result['metadatas'][0]
    distances = result['distances'][0]

    print(f"\n▶ {LABELS.get(col_name, col_name)}")
    print('-' * 50)
    for doc, meta, dist in zip(docs, metas, distances):
        etiqueta = meta.get('marca', meta.get('bloque', col_name))
        print(f'  [{dist:.3f}]  {etiqueta}')
        print(f'           {doc[:120]}…')


## Exportar a GH Pages — `docs/onfalo/cuadernos/`

Genera los HTMLs estáticos y actualiza `docs/_data/cuadernos_onfalo.yml`
para que el catálogo los muestre dinámicamente sin edición manual.

In [ ]:
import os
import yaml
from datetime import date

EXPORT_DIR = r'C:\Users\aleph\OASIS\aleph-scriptorium\DocumentMachineSDK\docs\onfalo\cuadernos'
DATA_FILE  = r'C:\Users\aleph\OASIS\aleph-scriptorium\DocumentMachineSDK\docs\_data\cuadernos_onfalo.yml'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Proyectos activos en esta ejecución (derivado de COLLECTIONS)
proyectos_activos = ', '.join(sorted({c.split('_')[1] for c in COLLECTIONS})).upper()

exports = {
    'corpus_2d': {
        'fig':         fig2d,
        'title':       'Espacio de embeddings 2D',
        'descripcion': f'Proyección UMAP 2D — proyectos activos: {proyectos_activos}. Paleta índigo — mod/onfalo.',
    },
    'corpus_3d': {
        'fig':         fig3d,
        'title':       'Espacio de embeddings 3D',
        'descripcion': f'Proyección UMAP 3D rotable — proyectos activos: {proyectos_activos}.',
    },
    'corpus_clusters': {
        'fig':         fig_cl,
        'title':       f'Clusters semánticos (K={best_k})',
        'descripcion': f'Agrupación K-means (K={best_k}) — proyectos activos: {proyectos_activos}.',
    },
}

# ── Exportar HTML ─────────────────────────────────────────────────────────────
for cuaderno_id, info in exports.items():
    fname = f'{cuaderno_id}.html'
    dest  = os.path.join(EXPORT_DIR, fname)
    info['fig'].write_html(dest, include_plotlyjs='cdn', full_html=True)
    print(f'✓ {fname}  ({os.path.getsize(dest) // 1024} KB)')

# ── Actualizar _data/cuadernos_onfalo.yml ─────────────────────────────────────
colecciones_str = ' · '.join(LABELS[c] for c in COLLECTIONS)
hoy = date.today().isoformat()

registros = []
for cuaderno_id, info in exports.items():
    registros.append({
        'id':          cuaderno_id,
        'title':       info['title'],
        'file':        f'onfalo/cuadernos/{cuaderno_id}.html',
        'descripcion': info['descripcion'],
        'fecha':       hoy,
        'colecciones': colecciones_str,
    })

header = (
    '# Cuadernos vectoriales — mod/onfalo\n'
    '# Generado/actualizado automáticamente por la celda de exportación del notebook.\n'
    '# Cada entrada se convierte en una card en el catálogo.\n\n'
)
with open(DATA_FILE, 'w', encoding='utf-8') as f:
    f.write(header)
    yaml.dump(registros, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print(f'\n✓ {DATA_FILE} actualizado ({len(registros)} cuadernos)')
print(f'  Proyectos activos: {proyectos_activos}')
